## 🔎 Benchmark Quality Check

In [1]:
import json
from pathlib import Path
import pandas as pd
import re
from collections import defaultdict

In [2]:
DATA_DIR = Path("/home/swiftie1230/EGO/FRAMING/FramingSensitivity/framing/outcome-oriented_framing/temporal_framing/output")

triage_path = DATA_DIR / "triage_allocation_temporal_framing_Qwen_Qwen2.5-7B-Instruct_5.jsonl"
unibench_path = DATA_DIR / "unibench_skeleton_temporal_framing_Qwen_Qwen2.5-7B-Instruct_5.jsonl"

In [9]:
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

triage = load_jsonl(triage_path)
unibench = load_jsonl(unibench_path)

len(triage), len(unibench)

data = triage + unibench
len(data)

10

In [10]:
def extract_rows(data):
    rows = []

    for ex in data:
        framings = ex["framings"]["temporal_focus"]["option_level"]

        for opt_id, content in framings.items():
            rows.append({
                "id": ex["id"],
                "dataset": ex["dataset"],
                "domain": ex["domain"],
                "option": opt_id,
                "source": content["source_text"],
                "short": content["short_term"],
                "long": content["long_term"],
            })

    return pd.DataFrame(rows)

triage_df = extract_rows(triage)
unibench_df = extract_rows(unibench)

df = pd.concat([triage_df, unibench_df], ignore_index=True)
df.head()

,id,dataset,domain,option,source,short,long
0,triage_alloc_00000,TRIAGE,life_safety,patient_1,Child F RR 0 Weak radial pulse Unresponsive Trapped under rubble; apneic after 5 rescue breaths,"Child F RR 0 Weak radial pulse Unresponsive Trapped under rubble; apneic after 5 rescue breaths, immediate medical intervention is required to stabilize the child.","""""""\nChild F RR 0 Weak radial pulse Unresponsive Trapped under rubble; apneic after 5 rescue breaths, with the likelihood of severe long-term neurological damage eventually manifesting in the coming months, impacting cognitive and motor functions significantly over time.\n"""""""
1,triage_alloc_00000,TRIAGE,life_safety,patient_2,"44-year-old female missing the distal aspect of her right upper extremity. She has the mangled, dirty extremity in a bag. Appears to be detached distal to the elbow. She has no tourniquet in place and is bleeding everywhere. She appears unsteady. Capillary refill is five seconds.","44-year-old female missing the distal aspect of her right upper extremity. The mangled, dirty extremity is in a bag. It appears to be detached distal to the elbow. She has no tourniquet in place and is bleeding profusely everywhere. She appears unsteady. Capillary refill is five seconds, indicating immediate blood loss.","44-year-old female missing the distal aspect of her right upper extremity. She has the mangled, dirty extremity in a bag. Appears to be detached distal to the elbow. In the long run, without proper medical intervention, she risks severe infection and potential amputation. Over time, she may also face significant psychological challenges and require extensive rehabilitation. Months from now, her quality of life could be severely impacted due to these long-term consequences."
2,triage_alloc_00001,TRIAGE,life_safety,patient_1,"7 y/o M RR 38 Absent distal pulse Groans, stops when spoken to Lying near bus","7 y/o M RR 38 Absent distal pulse Groans, stops when spoken to Lying near bus, immediate medical intervention required to stabilize condition.","7 y/o M RR 38 Absent distal pulse Groans, stops when spoken to Lying near bus. In the long run, this patient's condition may deteriorate without intervention, leading to potential limb loss and chronic pain over time."
3,triage_alloc_00001,TRIAGE,life_safety,patient_2,45 y/o F RR 0 Pulseless Unresponsive Driver of rear bus found under front of bus,45 y/o F RR 0 Pulseless Unresponsive Driver of rear bus found under front of bus; immediate CPR and stabilization efforts are required to prevent cardiac arrest.,"45 y/o F RR 0 Pulseless Unresponsive Driver of rear bus found under front of bus; in the long run, the emotional trauma and psychological impact on the driver’s family and colleagues will continue to be significant, potentially leading to long-lasting mental health issues and requiring ongoing support services."
4,triage_alloc_00002,TRIAGE,life_safety,patient_1,56-year-old male with burns to his entire body. He is unable to ambulate. You can’t assess capillary refill due to burns. Soot in nares with circumferential extremity burns. Complaining of extreme pain and begging for pain medication. Asking if he is going to die.,56-year-old male with burns to his entire body. He is unable to ambulate. You can’t assess capillary refill due to burns. Soot in nares with circumferential extremity burns. Experiencing extreme pain and begging for pain medication right now. Asking if he is going to die immediately.,"56-year-old male with burns to his entire body. He is unable to ambulate. In the long run, the extensive burns will lead to significant scarring and potential chronic pain, impacting his quality of life over the coming months and years. Soot in nares with circumferential extremity burns. As he recovers, the psychological trauma from the experience will persist, affecting his mental health and daily functioning. Complaining of extreme pain and begging for pain medication. Eventually, the need for ongoing pain mana

In [11]:
def get_base_options(example):
    options = {}

    for item in example["bundle"]["items"]:
        options[item["item_id"]] = item.get("text") or item.get("situation_text")

    return options

In [16]:
rows = []

for ex in data:
    base_opts = get_base_options(ex)
    framings = ex["framings"]["temporal_focus"]["option_level"]

    for opt_id in base_opts.keys():

        base = base_opts[opt_id]
        short = framings[opt_id]["short_term"].replace('"""', "").strip()
        long  = framings[opt_id]["long_term"].replace('"""', "").strip()

        rows.append({
            "id": ex["id"],
            "dataset": ex["dataset"],
            "domain": ex["domain"],
            "option": opt_id,

            "BASE": base,
            "SHORT": short,
            "LONG": long,
        })

df = pd.DataFrame(rows)
df

,id,dataset,domain,option,BASE,SHORT,LONG
0,triage_alloc_00000,TRIAGE,life_safety,patient_1,Child F RR 0 Weak radial pulse Unresponsive Trapped under rubble; apneic after 5 rescue breaths,"Child F RR 0 Weak radial pulse Unresponsive Trapped under rubble; apneic after 5 rescue breaths, immediate medical intervention is required to stabilize the child.","Child F RR 0 Weak radial pulse Unresponsive Trapped under rubble; apneic after 5 rescue breaths, with the likelihood of severe long-term neurological damage eventually manifesting in the coming months, impacting cognitive and motor functions significantly over time."
1,triage_alloc_00000,TRIAGE,life_safety,patient_2,"44-year-old female missing the distal aspect of her right upper extremity. She has the mangled, dirty extremity in a bag. Appears to be detached distal to the elbow. She has no tourniquet in place and is bleeding everywhere. She appears unsteady. Capillary refill is five seconds.","44-year-old female missing the distal aspect of her right upper extremity. The mangled, dirty extremity is in a bag. It appears to be detached distal to the elbow. She has no tourniquet in place and is bleeding profusely everywhere. She appears unsteady. Capillary refill is five seconds, indicating immediate blood loss.","44-year-old female missing the distal aspect of her right upper extremity. She has the mangled, dirty extremity in a bag. Appears to be detached distal to the elbow. In the long run, without proper medical intervention, she risks severe infection and potential amputation. Over time, she may also face significant psychological challenges and require extensive rehabilitation. Months from now, her quality of life could be severely impacted due to these long-term consequences."
2,triage_alloc_00001,TRIAGE,life_safety,patient_1,"7 y/o M RR 38 Absent distal pulse Groans, stops when spoken to Lying near bus","7 y/o M RR 38 Absent distal pulse Groans, stops when spoken to Lying near bus, immediate medical intervention required to stabilize condition.","7 y/o M RR 38 Absent distal pulse Groans, stops when spoken to Lying near bus. In the long run, this patient's condition may deteriorate without intervention, leading to potential limb loss and chronic pain over time."
3,triage_alloc_00001,TRIAGE,life_safety,patient_2,45 y/o F RR 0 Pulseless Unresponsive Driver of rear bus found under front of bus,45 y/o F RR 0 Pulseless Unresponsive Driver of rear bus found under front of bus; immediate CPR and stabilization efforts are required to prevent cardiac arrest.,"45 y/o F RR 0 Pulseless Unresponsive Driver of rear bus found under front of bus; in the long run, the emotional trauma and psychological impact on the driver’s family and colleagues will continue to be significant, potentially leading to long-lasting mental health issues and requiring ongoing support services."
4,triage_alloc_00002,TRIAGE,life_safety,patient_1,56-year-old male with burns to his entire body. He is unable to ambulate. You can’t assess capillary refill due to burns. Soot in nares with circumferential extremity burns. Complaining of extreme pain and begging for pain medication. Asking if he is going to die.,56-year-old male with burns to his entire body. He is unable to ambulate. You can’t assess capillary refill due to burns. Soot in nares with circumferential extremity burns. Experiencing extreme pain and begging for pain medication right now. Asking if he is going to die immediately.,"56-year-old male with burns to his entire body. He is unable to ambulate. In the long run, the extensive burns will lead to significant scarring and potential chronic pain, impacting his quality of life over the coming months and years. Soot in nares with circumferential extremity burns. As he recovers, the psychological trauma from the experience will persist, affecting his mental health and daily functioning. Complaining of extreme pain and begging for pain medication. Eventually, the need for ongoing pain management will become

In [13]:
df = df.sort_values(["dataset", "id", "option"])

In [15]:
pd.set_option("display.max_colwidth", None)
df

,id,dataset,domain,option,BASE,SHORT,LONG
0,triage_alloc_00000,TRIAGE,life_safety,patient_1,Child F RR 0 Weak radial pulse Unresponsive Trapped under rubble; apneic after 5 rescue breaths,"Child F RR 0 Weak radial pulse Unresponsive Trapped under rubble; apneic after 5 rescue breaths, immediate medical intervention is required to stabilize the child.","Child F RR 0 Weak radial pulse Unresponsive Trapped under rubble; apneic after 5 rescue breaths, with the likelihood of severe long-term neurological damage eventually manifesting in the coming months, impacting cognitive and motor functions significantly over time."
1,triage_alloc_00000,TRIAGE,life_safety,patient_2,"44-year-old female missing the distal aspect of her right upper extremity. She has the mangled, dirty extremity in a bag. Appears to be detached distal to the elbow. She has no tourniquet in place and is bleeding everywhere. She appears unsteady. Capillary refill is five seconds.","44-year-old female missing the distal aspect of her right upper extremity. The mangled, dirty extremity is in a bag. It appears to be detached distal to the elbow. She has no tourniquet in place and is bleeding profusely everywhere. She appears unsteady. Capillary refill is five seconds, indicating immediate blood loss.","44-year-old female missing the distal aspect of her right upper extremity. She has the mangled, dirty extremity in a bag. Appears to be detached distal to the elbow. In the long run, without proper medical intervention, she risks severe infection and potential amputation. Over time, she may also face significant psychological challenges and require extensive rehabilitation. Months from now, her quality of life could be severely impacted due to these long-term consequences."
2,triage_alloc_00001,TRIAGE,life_safety,patient_1,"7 y/o M RR 38 Absent distal pulse Groans, stops when spoken to Lying near bus","7 y/o M RR 38 Absent distal pulse Groans, stops when spoken to Lying near bus, immediate medical intervention required to stabilize condition.","7 y/o M RR 38 Absent distal pulse Groans, stops when spoken to Lying near bus. In the long run, this patient's condition may deteriorate without intervention, leading to potential limb loss and chronic pain over time."
3,triage_alloc_00001,TRIAGE,life_safety,patient_2,45 y/o F RR 0 Pulseless Unresponsive Driver of rear bus found under front of bus,45 y/o F RR 0 Pulseless Unresponsive Driver of rear bus found under front of bus; immediate CPR and stabilization efforts are required to prevent cardiac arrest.,"45 y/o F RR 0 Pulseless Unresponsive Driver of rear bus found under front of bus; in the long run, the emotional trauma and psychological impact on the driver’s family and colleagues will continue to be significant, potentially leading to long-lasting mental health issues and requiring ongoing support services."
4,triage_alloc_00002,TRIAGE,life_safety,patient_1,56-year-old male with burns to his entire body. He is unable to ambulate. You can’t assess capillary refill due to burns. Soot in nares with circumferential extremity burns. Complaining of extreme pain and begging for pain medication. Asking if he is going to die.,56-year-old male with burns to his entire body. He is unable to ambulate. You can’t assess capillary refill due to burns. Soot in nares with circumferential extremity burns. Experiencing extreme pain and begging for pain medication right now. Asking if he is going to die immediately.,"56-year-old male with burns to his entire body. He is unable to ambulate. In the long run, the extensive burns will lead to significant scarring and potential chronic pain, impacting his quality of life over the coming months and years. Soot in nares with circumferential extremity burns. As he recovers, the psychological trauma from the experience will persist, affecting his mental health and daily functioning. Complaining of extreme pain and begging for pain medication. Eventually, the need for ongoing pain management will become